# Simulation-Based Inference via Neural Ratio Estimation (NRE)

**Model:**  $X \mid \theta \sim \mathcal{N}(\theta,\, \sigma^2_{\text{lik}})$,  $\theta \sim \mathcal{N}(0,\, \sigma^2_0)$

**Goal:** Given observed $x$, construct a confidence set for $\theta$ — without ever evaluating the likelihood directly.

**Key idea:** Train a binary classifier to distinguish samples from the **joint** $p(\theta, X)$ vs. the **marginal product** $p(\theta)p(X)$.  
The optimal classifier output satisfies:
$$\frac{d(\theta,x)}{1-d(\theta,x)} = \frac{p(\theta,x)}{p(\theta)p(x)} = \frac{p(x\mid\theta)}{p(x)} =: r(\theta, x)$$

This likelihood ratio is all we need for both Bayesian and frequentist inference.

---
**Two routes:**

| Route | Method | Guarantee |
|-------|--------|-----------|
| Bayesian | $\text{posterior} \propto \hat{r}(\theta, x)\cdot p(\theta)$ → HPD interval | Posterior credible interval |
| Frequentist | $T(\theta,x) = -\log\hat{r}(\theta,x)$ → Neyman inversion | Frequentist coverage |

Since this is a Gaussian model, analytical solutions exist for both — we can verify the classifier-based results directly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, interpolate
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

np.random.seed(2025)

## 1. Model Settings and Analytical Solutions

For this Gaussian model the posterior is also Gaussian (conjugate prior):
$$\theta \mid X \sim \mathcal{N}\!\left(\mu_{\text{post}},\, \sigma^2_{\text{post}}\right), \quad
\sigma^2_{\text{post}} = \frac{1}{1/\sigma^2_0 + 1/\sigma^2_{\text{lik}}}, \quad
\mu_{\text{post}} = \sigma^2_{\text{post}}\cdot\frac{x}{\sigma^2_{\text{lik}}}$$

The frequentist confidence interval from the likelihood ratio test is:
$$\text{CS}(x) = \left\{\theta : \frac{(x-\theta)^2}{\sigma^2_{\text{lik}}} \leq \chi^2_{1,1-\alpha}\right\} = x \pm \sqrt{\sigma^2_{\text{lik}}\cdot\chi^2_{1,1-\alpha}}$$

In [ ]:
# ── Model parameters ─────────────────────────────────────────────────────────
SIGMA2_LIK   = 0.25   # likelihood variance: keeps log-ratio in [-8, 1], learnable by MLP
MU0          = 0.0
SIGMA2_0     = 0.5    # prior variance
PARAM_LO     = -1.5
PARAM_HI     =  1.5
ALPHA        = 0.10   # → 90% confidence

# ── Simulation sizes ─────────────────────────────────────────────────────────
B_TRAIN = 10_000   # classifier training samples
B_CAL   =  2_000   # per-θ calibration samples (frequentist critical values)
B_COVER =    500   # samples for coverage diagnostics

# ── Derived quantities ────────────────────────────────────────────────────────
SIGMA2_MARG = SIGMA2_LIK + SIGMA2_0          # marginal p(X) = N(0, σ²_lik + σ²_0)
SIGMA2_POST = 1 / (1/SIGMA2_0 + 1/SIGMA2_LIK)

def posterior_mean(x):
    return SIGMA2_POST * x / SIGMA2_LIK

def analytical_bayes_ci(x):
    """90% equal-tail posterior credible interval."""
    mu = posterior_mean(x)
    z  = stats.norm.ppf(1 - ALPHA/2)
    return mu - z*np.sqrt(SIGMA2_POST), mu + z*np.sqrt(SIGMA2_POST)

def analytical_freq_ci(x):
    """90% frequentist CI via LRT inversion."""
    hw = np.sqrt(SIGMA2_LIK * stats.chi2.ppf(1 - ALPHA, df=1))
    return x - hw, x + hw

print(f"Model:  X|θ ~ N(θ, {SIGMA2_LIK}),  θ ~ N({MU0}, {SIGMA2_0})")
print(f"Marginal X ~ N(0, {SIGMA2_MARG})")
print(f"Posterior σ²_post = {SIGMA2_POST:.5f}")
print()
x_obs = 0.5
lo_b, hi_b = analytical_bayes_ci(x_obs)
lo_f, hi_f = analytical_freq_ci(x_obs)
print(f"For x_obs = {x_obs}:")
print(f"  Analytical 90% Bayesian CI:     ({lo_b:.4f}, {hi_b:.4f})")
print(f"  Analytical 90% frequentist CI:  ({lo_f:.4f}, {hi_f:.4f})")

## 2. Generate Classifier Training Data

We need samples from two distributions:
- **Positive (label 1):** $(\theta, X)$ from the joint $p(\theta)p(X\mid\theta)$
- **Negative (label 0):** $(\theta, X)$ from the product of marginals $p(\theta)p(X)$

The simplest way to get negatives: sample the joint, then **shuffle** $X$ to break the dependence on $\theta$.

In [ ]:
def sample_joint(n):
    theta = np.random.normal(MU0, np.sqrt(SIGMA2_0), n)
    x     = np.random.normal(theta, np.sqrt(SIGMA2_LIK))
    return theta, x

# Positive: joint samples
theta_pos, x_pos = sample_joint(B_TRAIN)

# Negative: shuffle X to decouple from θ
theta_neg = theta_pos.copy()
x_neg     = np.random.permutation(x_pos)

features = np.column_stack([
    np.concatenate([theta_pos, theta_neg]),
    np.concatenate([x_pos,     x_neg])
])
labels = np.concatenate([np.ones(B_TRAIN), np.zeros(B_TRAIN)])

perm     = np.random.permutation(len(labels))
features, labels = features[perm], labels[perm]

print(f"Training set: {len(labels):,} total  |  {int(labels.sum()):,} joint  |  {int((1-labels).sum()):,} marginal product")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(features[labels==1, 0], features[labels==1, 1], alpha=0.1, s=5, c='steelblue', label='Joint (label=1)')
axes[0].scatter(features[labels==0, 0], features[labels==0, 1], alpha=0.1, s=5, c='tomato',    label='Product (label=0)')
axes[0].set(xlabel='θ', ylabel='X', title='Training Data')
axes[0].legend(markerscale=4)
axes[1].hist(features[labels==1, 1], bins=50, alpha=0.5, density=True, label='X from joint')
axes[1].hist(features[labels==0, 1], bins=50, alpha=0.5, density=True, label='X shuffled')
axes[1].set(xlabel='X', title='Marginal X distribution')
axes[1].legend()
plt.tight_layout()
plt.show()

## 3. Train the Classifier

A 2-layer MLP takes $(\theta, X)$ as input and predicts $d(\theta, X) = P(\text{joint} \mid \theta, X)$.

From the optimal classifier, we recover the **log likelihood ratio**:
$$\log \hat{r}(\theta, x) = \log\frac{d(\theta,x)}{1 - d(\theta,x)}$$

In [ ]:
scaler = StandardScaler().fit(features)

clf = MLPClassifier(
    hidden_layer_sizes=(64, 64),
    activation='relu',
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=2025
)
clf.fit(scaler.transform(features), labels)
print(f"Validation accuracy: {clf.best_validation_score_:.4f}")

def log_ratio_hat(theta_arr, x_arr):
    """log r̂(θ, x) = log d/(1-d)."""
    inp = scaler.transform(np.column_stack([np.atleast_1d(theta_arr), np.atleast_1d(x_arr)]))
    d   = clf.predict_proba(inp)[:, 1].clip(1e-6, 1-1e-6)
    return np.log(d / (1 - d))

## 4. Verify: Estimated vs True Log-Ratio

For our Gaussian model the true log-ratio is:
$$\log r(\theta, x) = \log p(x\mid\theta) - \log p(x)
= -\frac{(x-\theta)^2}{2\sigma^2_{\text{lik}}} + \frac{x^2}{2\sigma^2_{\text{marg}}} + C$$

We check that the classifier learned this correctly.

In [ ]:
theta_g = np.linspace(PARAM_LO, PARAM_HI, 300)

true_log_r = (stats.norm.logpdf(x_obs, theta_g, np.sqrt(SIGMA2_LIK))
              - stats.norm.logpdf(x_obs, 0.0,    np.sqrt(SIGMA2_MARG)))
est_log_r  = log_ratio_hat(theta_g, np.full_like(theta_g, x_obs))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(theta_g, true_log_r, 'k-',  lw=2, label='True log r(θ, x)')
ax.plot(theta_g, est_log_r,  'r--', lw=2, label='Classifier estimate')
ax.axhline(0, color='gray', ls=':', lw=1)
ax.axvline(x_obs, color='steelblue', ls='--', alpha=0.6, label=f'x_obs = {x_obs}')
ax.set(xlabel='θ', ylabel='log r(θ, x)', title=f'Log-Likelihood Ratio Verification  (x_obs = {x_obs})')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Bayesian Route: Classifier-Based Posterior

By Bayes' theorem:
$$p(\theta \mid x) \propto p(x\mid\theta)\cdot p(\theta) = r(\theta,x)\cdot p(x)\cdot p(\theta) \propto r(\theta,x)\cdot p(\theta)$$

So we approximate the posterior as:
$$\hat{p}(\theta\mid x) \propto \hat{r}(\theta, x) \cdot p(\theta)$$

Then find the **90% Highest Posterior Density (HPD) interval**: the smallest region containing 90% of the posterior mass.

In [ ]:
theta_fine = np.linspace(PARAM_LO, PARAM_HI, 1000)
dtheta     = theta_fine[1] - theta_fine[0]

# Unnormalized log-posterior
log_r      = log_ratio_hat(theta_fine, np.full_like(theta_fine, x_obs))
log_prior  = stats.norm.logpdf(theta_fine, MU0, np.sqrt(SIGMA2_0))
log_post   = log_r + log_prior
log_post  -= log_post.max()          # shift for numerical stability
post       = np.exp(log_post)
post      /= post.sum() * dtheta     # normalize

# 90% HPD: include highest-density points until we accumulate 90% mass
sorted_idx = np.argsort(-post)
cumulative, hpd_mask = 0.0, np.zeros(len(theta_fine), dtype=bool)
for i in sorted_idx:
    hpd_mask[i] = True
    cumulative += post[i] * dtheta
    if cumulative >= 1 - ALPHA:
        break

true_post  = stats.norm.pdf(theta_fine, posterior_mean(x_obs), np.sqrt(SIGMA2_POST))
lo_a, hi_a = analytical_bayes_ci(x_obs)
hpd_lo     = theta_fine[hpd_mask].min()
hpd_hi     = theta_fine[hpd_mask].max()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(theta_fine, true_post, 'k-',  lw=2, label='True posterior')
ax.plot(theta_fine, post,      'r--', lw=2, label='Classifier-based posterior')
ax.fill_between(theta_fine, post, where=hpd_mask, alpha=0.25, color='red',
                label=f'90% HPD (classifier): ({hpd_lo:.3f}, {hpd_hi:.3f})')
ax.axvline(lo_a, color='k', ls=':', lw=1.5)
ax.axvline(hi_a, color='k', ls=':', lw=1.5,
           label=f'Analytical 90% CI: ({lo_a:.3f}, {hi_a:.3f})')
ax.axvline(x_obs, color='steelblue', ls='--', alpha=0.7, label=f'x_obs = {x_obs}')
ax.set(xlabel='θ', ylabel='Density', title='Bayesian Route: Classifier-Based Posterior')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Classifier 90% HPD:  ({hpd_lo:.4f}, {hpd_hi:.4f})")
print(f"Analytical 90% CI:   ({lo_a:.4f}, {hi_a:.4f})")

## 6. Frequentist Route: Neyman Inversion

Define the test statistic:
$$T(\theta, x) = -\log \hat{r}(\theta, x)$$

Large $T$ means $\theta$ is poorly supported by $x$ (reject $H_0\colon\theta$ is true).  
Small $T$ (or negative) means $\theta$ is well-supported.

**Critical value calibration:** For each $\theta$ on a grid, simulate $X^* \mid \theta$, compute $T(\theta, X^*)$, and take the 90th percentile.

**Confidence set:** $\text{CS}(x) = \{\theta : T(\theta, x) \leq c_\alpha(\theta)\}$

In [ ]:
def test_stat(theta_arr, x_arr):
    """T(θ, x) = -log r̂(θ, x).  Large value → reject H₀: θ is true."""
    return -log_ratio_hat(np.atleast_1d(theta_arr), np.atleast_1d(x_arr))

# Calibrate critical values on a θ grid
theta_cal = np.linspace(PARAM_LO, PARAM_HI, 40)
crit_vals = np.zeros(len(theta_cal))

for i, th in enumerate(theta_cal):
    x_star       = np.random.normal(th, np.sqrt(SIGMA2_LIK), B_CAL)
    t_vals       = test_stat(np.full(B_CAL, th), x_star)
    crit_vals[i] = np.quantile(t_vals, 1 - ALPHA)

c_alpha = interpolate.interp1d(theta_cal, crit_vals, kind='linear', fill_value='extrapolate')

# Reference: if classifier is perfect, T ≈ LRT/2 ~ χ²(1)/2 under H₀
chi2_ref = stats.chi2.ppf(1 - ALPHA, df=1) / 2

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(theta_cal, crit_vals, 'bo-', ms=4, label='Empirical $c_\\alpha(\\theta)$')
ax.axhline(chi2_ref, color='gray', ls='--', label=f'$\\chi^2_{{1,0.90}}/2 = {chi2_ref:.3f}$ (reference)')
ax.set(xlabel='θ', ylabel='Critical value', title='Calibrated Critical Values $c_\\alpha(\\theta)$')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Confidence set for x_obs via Neyman inversion
theta_scan = np.linspace(PARAM_LO, PARAM_HI, 500)
T_obs      = test_stat(theta_scan, np.full_like(theta_scan, x_obs))
C_vals     = c_alpha(theta_scan)
in_cs      = T_obs <= C_vals

lo_cs = theta_scan[in_cs].min() if in_cs.any() else np.nan
hi_cs = theta_scan[in_cs].max() if in_cs.any() else np.nan
lo_f, hi_f = analytical_freq_ci(x_obs)

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

ax = axes[0]
ax.plot(theta_scan, T_obs,  'b-',  lw=2, label='$T(\\theta, x_{\\mathrm{obs}}) = -\\log\\hat{r}(\\theta, x_{\\mathrm{obs}})$')
ax.plot(theta_scan, C_vals, 'r--', lw=2, label='$c_\\alpha(\\theta)$')
ax.fill_between(theta_scan, T_obs, C_vals, where=in_cs, alpha=0.2, color='green', label='CS region')
ax.axhline(0, color='gray', ls=':', lw=1)
ax.set(ylabel='Test statistic', title='Frequentist Route: Neyman Inversion')
ax.legend(fontsize=9)

ax = axes[1]
ax.barh([1, 0], [hi_cs - lo_cs, hi_f - lo_f], left=[lo_cs, lo_f],
        height=0.4, color=['green', 'gray'], alpha=0.6)
ax.axvline(x_obs, color='steelblue', ls='--', label=f'$x_{{\\mathrm{{obs}}}} = {x_obs}$')
ax.set_yticks([0, 1])
ax.set_yticklabels(['Analytical LRT CI', 'Classifier CS'])
ax.set(xlabel='θ', xlim=(PARAM_LO, PARAM_HI), title='90% Confidence Sets')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Classifier CS  90%:  ({lo_cs:.4f}, {hi_cs:.4f})")
print(f"Analytical CI  90%:  ({lo_f:.4f}, {hi_f:.4f})")

## 7. Summary Comparison

Side-by-side: Bayesian HPD vs Frequentist CS, both classifier-based and analytical.

In [ ]:
lo_b_clf, hi_b_clf   = hpd_lo, hpd_hi
lo_b_ana, hi_b_ana   = analytical_bayes_ci(x_obs)
lo_f_clf, hi_f_clf   = lo_cs, hi_cs
lo_f_ana, hi_f_ana   = analytical_freq_ci(x_obs)

labels_plot  = ['Bayesian (analytical)', 'Bayesian (classifier)', 'Frequentist (analytical)', 'Frequentist (classifier)']
los          = [lo_b_ana, lo_b_clf, lo_f_ana, lo_f_clf]
his          = [hi_b_ana, hi_b_clf, hi_f_ana, hi_f_clf]
colors       = ['#2196F3', '#64B5F6', '#4CAF50', '#81C784']

fig, ax = plt.subplots(figsize=(9, 3))
for i, (lo, hi, lbl, col) in enumerate(zip(los, his, labels_plot, colors)):
    ax.barh(i, hi - lo, left=lo, height=0.5, color=col, alpha=0.8, label=f'{lbl}: ({lo:.3f}, {hi:.3f})')
ax.axvline(x_obs, color='k', ls='--', lw=1.5, label=f'$x_{{obs}} = {x_obs}$')
ax.set_yticks(range(4))
ax.set_yticklabels(labels_plot)
ax.set(xlabel='θ', title=f'90% Intervals for $x_{{obs}} = {x_obs}$', xlim=(PARAM_LO, PARAM_HI))
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## 8. Coverage Diagnostics

To verify the confidence guarantees, we simulate many test pairs $(\theta, x) \sim p(\theta, X)$ and check whether each true $\theta$ falls in the constructed interval.

- **Frequentist coverage:** should be $\geq 90\%$ (by construction via Neyman inversion)
- **Bayesian coverage:** should be approximately $90\%$ (posterior credible intervals have approximate frequentist coverage in this conjugate model)

In [ ]:
theta_test, x_test = sample_joint(B_COVER)

# ── Frequentist coverage ─────────────────────────────────────────────────────
T_test       = test_stat(theta_test, x_test)
c_test       = c_alpha(theta_test.clip(PARAM_LO, PARAM_HI))
freq_covered = T_test <= c_test

# ── Bayesian coverage (vectorized over test set) ──────────────────────────────
theta_grid_cov = np.linspace(PARAM_LO, PARAM_HI, 200)
dth            = theta_grid_cov[1] - theta_grid_cov[0]
M, N           = len(theta_grid_cov), B_COVER

# Evaluate classifier on all (theta_grid[j], x_test[i]) pairs at once
theta_rep  = np.tile(theta_grid_cov, N)     # (N*M,)
x_rep      = np.repeat(x_test, M)           # (N*M,)
log_r_flat = log_ratio_hat(theta_rep, x_rep)
log_r_mat  = log_r_flat.reshape(N, M)       # (N, M)

log_prior_g  = stats.norm.logpdf(theta_grid_cov, MU0, np.sqrt(SIGMA2_0))
log_post_mat = log_r_mat + log_prior_g[np.newaxis, :]
log_post_mat -= log_post_mat.max(axis=1, keepdims=True)
post_mat     = np.exp(log_post_mat)
post_mat    /= post_mat.sum(axis=1, keepdims=True) * dth

# Vectorized HPD check
sorted_post  = np.sort(post_mat, axis=1)[:, ::-1]          # (N, M) descending
cumsum_post  = np.cumsum(sorted_post, axis=1) * dth        # (N, M)
thresh_idx   = np.minimum((cumsum_post < 1 - ALPHA).sum(axis=1), M - 1)
thresholds   = sorted_post[np.arange(N), thresh_idx]       # density cutoff per test sample

# Check: is post(theta_test[i] | x_test[i]) above the HPD threshold?
test_idx       = np.argmin(np.abs(theta_grid_cov[:, None] - theta_test[None, :]), axis=0)
density_test   = post_mat[np.arange(N), test_idx]
bayes_covered  = density_test >= thresholds

print("=" * 40)
print(f"Coverage Diagnostics (n = {B_COVER})")
print(f"Target: {(1-ALPHA)*100:.0f}%")
print(f"Frequentist (classifier CS):  {freq_covered.mean()*100:.1f}%")
print(f"Bayesian   (classifier HPD):  {bayes_covered.mean()*100:.1f}%")
print("=" * 40)

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(['Frequentist\n(classifier CS)', 'Bayesian\n(classifier HPD)'],
       [freq_covered.mean()*100, bayes_covered.mean()*100],
       color=['#4CAF50', '#2196F3'], alpha=0.8, width=0.4)
ax.axhline((1-ALPHA)*100, color='k', ls='--', lw=1.5, label=f'Target {(1-ALPHA)*100:.0f}%')
ax.set(ylabel='Coverage (%)', title=f'Empirical Coverage (n = {B_COVER})', ylim=(70, 100))
ax.legend()
plt.tight_layout()
plt.show()